In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from scipy.interpolate import CubicSpline

m, w, lam, QA, QB, T = 1.0, 1.0, 1.0, 1.0, -0.5, 2.0
dV  = lambda q: w**2 * q + lam * q**3
rhs = lambda t, y: [y[1], -dV(y[0]) / m]

def miss(v0):
    s = solve_ivp(rhs, (0, T), [QA, v0], rtol=1e-13, atol=1e-15)
    return s.y[0, -1] - QB

v0_star = brentq(miss, -4.0, -1.0, xtol=1e-15, rtol=8.9e-16)   # fundamental branch
sol     = solve_ivp(rhs, (0, T), [QA, v0_star], rtol=1e-13, atol=1e-15,
                    dense_output=True)
t_ref   = np.linspace(0, T, 20001)
q_true  = CubicSpline(t_ref, sol.sol(t_ref)[0])
print(f"v0* = {v0_star:.15f}")

In [ ]:
# one degree of freedom, anharmonic well:  L = ½ q̇² − V(q)
V  = lambda q: 0.5*q**2 + 0.25*q**4
dV = lambda q: q + q**3
N = 200

def action(q, h):
    qd = np.diff(q)/h                       # velocities on cell centers
    Vbar = 0.5*(V(q[:-1]) + V(q[1:]))       # trapezoid for the potential
    return np.sum(0.5*qd**2 - Vbar)*h

t  = np.linspace(0.0, T, N + 1)
h = t[1] - t[0]
q_guess = np.linspace(QA, QB, N + 1)          # not np.linspace(1.0, -5.0, ...)

assert np.isclose(q_guess[0],  QA) and np.isclose(q_guess[-1], QB), \
    "initial guess endpoints do not match the boundary conditions"
S0 = action(q_guess, h)
print("S0[straight line =", S0)

In [ ]:
shape_smooth = np.sin(np.pi*t/T)             # well-behaved mode

rng = np.random.default_rng(0)               # deliberately ugly
shape_random = rng.normal(size=t.size)
shape_random[0] = shape_random[-1] = 0.0     # MUST vanish at the endpoints
shape_random /= np.abs(shape_random).max()

for shape in (shape_smooth, shape_random):
    for eps in (0.2, 0.1, 0.05, 0.025):
        dS = action(q_true(t)+ eps*shape, h) - S0
        print(f"eps={eps:<7} dS={dS: .6e}  dS/eps={dS/eps: .4f}")

In [ ]:
# q_true: the exact path with q(0)=0, q(1)=0.5  (provided in the notebook)
S0 = action(q_true(t), h)
print("S0[exact path =", S0)
shape_smooth = np.sin(np.pi*t/T)             # well-behaved mode

rng = np.random.default_rng(0)               # deliberately ugly
shape_random = rng.normal(size=t.size)
shape_random[0] = shape_random[-1] = 0.0     # MUST vanish at the endpoints
shape_random /= np.abs(shape_random).max()

for shape in (shape_smooth, shape_random):
    for eps in (0.2, 0.1, 0.05, 0.025):
        dS = action(q_true(t)+ eps*shape, h) - S0
        print(f"eps={eps:<7} dS={dS: .6e}  dS/eps={dS/eps: .4f} dS/eps^2={dS/eps**2: .4f}")

In [ ]:
Vh = lambda q: 0.5*q**2
def action_h(q, h):
    qd = np.diff(q)/h
    Vbar = 0.5*(Vh(q[:-1]) + Vh(q[1:]))
    return np.sum(0.5*qd**2 - Vbar)*h

eps = 1e-3
for T in (1.0, 2.0, 3.0, np.pi, 3.5, 4.0, 5.0):
    t = np.linspace(0, T, 4001); h = t[1]-t[0]
    q = eps*np.sin(np.pi*t/T)                # true path is q ≡ 0
    print(f"T={T:<8.5f}  S/eps^2 = {action_h(q,h)/eps**2: .6f}")